# 01 Data Preparation

This notebook loads a timestamped corpus, assigns time periods, tokenizes texts, and saves a processed file for embedding training.

In [ ]:
from pathlib import Path
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(PROJECT_ROOT))

from src.preprocessing import DEFAULT_PERIODS, load_corpus, preprocess_dataframe, sentences_by_period

In [ ]:
RAW_PATH = PROJECT_ROOT / "data" / "raw" / "corpus.csv"
PROCESSED_PATH = PROJECT_ROOT / "data" / "processed" / "corpus_tokenized.pkl"

if not RAW_PATH.exists():
    raise FileNotFoundError(
        f"Expected {RAW_PATH}. Create a CSV with columns: year,text"
    )

In [ ]:
df = load_corpus(RAW_PATH)
df.head()

In [ ]:
processed = preprocess_dataframe(df, periods=DEFAULT_PERIODS, min_tokens=5)
processed[["year", "period", "tokens"]].head()

In [ ]:
summary = (
    processed.assign(token_count=processed["tokens"].map(len))
    .groupby("period", as_index=False)
    .agg(documents=("text", "size"), tokens=("token_count", "sum"), first_year=("year", "min"), last_year=("year", "max"))
)
summary

In [ ]:
PROCESSED_PATH.parent.mkdir(parents=True, exist_ok=True)
processed.to_pickle(PROCESSED_PATH)
PROCESSED_PATH